In [22]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 


In [23]:
## LEARNING IMSBS
#m2_results = pd.read_csv("results_16_train_instances_m2/LIMSBS-500_imbs-iters-3000_num-roots-10_U10_5_5_A2_F2-1.csv")
#m3_results = pd.read_csv("results_16_train_instances_m3/LIMSBS-500_imbs-iters-3000_num-roots-10_U10_10_5_A2_F2-3.csv") # (the best for now) LIMSBS-500_imbs-iters-3000_num-roots-10_U10_10_5_A2_F3.csv"
#m5_results = pd.read_csv("results_16_train_instances_m5/LIMSBS-500_imbs-iters-3000_num-roots-10_U10_5_5_A3_F1-1.csv")
#m10_results = pd.read_csv("results_16_train_instances_m10/LIMSBS-500_imbs-iters-3000_num-roots-10_U10_5_5_A2_F2-1.csv")   # (lamda=0.45 everywhere) LIMSBS-500_imbs-iters-3000_num-roots-10_U10_10_10_A3_F1.csv")  #LIMSBS-500_imbs-iters-3000_num-roots-10_U10_5_5_A2_F3.csv")

## LEARNING IMSBS (after tuning):

m2_results = pd.read_csv("tuned_results/ensemble/results_16_train_instances_m2/LIMSBS-2000_imbs-iters-5000_num-roots-10_U10_5_5_A2_F2.csv")
m3_results = pd.read_csv("tuned_results/ensemble/results_16_train_instances_m3/LIMSBS-2000_imbs-iters-5000_num-roots-10_U10_10_5_A2_F2-1.csv") # (the best for now) LIMSBS-500_imbs-iters-3000_num-roots-10_U10_10_5_A2_F3.csv"

m5_results = pd.read_csv("tuned_results/ensemble/results_16_train_instances_m5/LIMSBS-2000_imbs-iters-5000_num-roots-10_U10_5_5_A3_F1.csv")

m10_results = pd.read_csv("tuned_results/ensemble/results_16_train_instances_m10/LIMSBS-2000_imbs-iters-5000_num-roots-10_U10_5_5_A2_F2.csv")   # (lamda=0.45 everywhere) LIMSBS-500_imbs-iters-3000_num-roots-10_U10_10_10_A3_F1.csv")  #LIMSBS-500_imbs-iters-3000_num-roots-10_U10_5_5_A2_F3.csv")
# TODO: update the results 

## IMSBS
IMSBS = pd.read_csv("tuned_results/IMSBS/IMSBS-2000_heuristic-h5_imbs-iters-5000_num-roots-10-tune.csv")

In [24]:
def format_and_aggregate(df):
    
    df["m"]=df.apply(lambda x: x["file"].split("_")[1], axis=1) 
    df["n"]=df.apply(lambda x: x["file"].split("_")[2], axis=1)
    df["sigma"]=df.apply(lambda x: x["file"].split("_")[3], axis=1)
    df["index"]=df.apply(lambda x: x["file"].split("_")[4].split(".")[0], axis = 1)

    df.drop(["file"], axis=1, inplace=True)
    
    #group by (m, n, sigma)
    df_results_group = df.groupby(by=["m", "n", "sigma"]).agg({"time": "mean", "quality" : "mean"} ).reset_index()
    #m2_results_group.drop(["index"], axis=1, inplace=True)

    return df_results_group[["m", "n", "sigma", "time", "quality"]] #_group


In [25]:
m2_results_group = format_and_aggregate(m2_results)
m3_results_group = format_and_aggregate(m3_results)
m5_results_group = format_and_aggregate(m5_results)
m10_results_group = format_and_aggregate(m10_results)

IMSBS_group = format_and_aggregate(IMSBS)
 

In [26]:
# filter
m2_results_group_filter  = m2_results_group[m2_results_group["m"] == "2"]
m3_results_group_filter  = m3_results_group[m3_results_group["m"] == "3"]
m5_results_group_filter  = m5_results_group[m5_results_group["m"] == "5"]
m10_results_group_filter = m10_results_group[m10_results_group["m"] == "10"]

# collect
results = [
    m2_results_group_filter,
    m3_results_group_filter,
    m5_results_group_filter,
    m10_results_group_filter
]

# union
results_limsbs = pd.concat(results, axis=0, ignore_index=True)
results_limsbs.rename(columns={"time": "time_limsbs", "quality": "quality_limsbs"}, inplace=True)

IMSBS_group
IMSBS_group.rename(columns={"time": "time_imsbs", "quality": "quality_imsbs"}, inplace=True)


In [27]:
#results
results_limsbs

,m,n,sigma,time_limsbs,quality_limsbs
0,2,100,2,0.629813,72.4
1,2,100,4,0.730760,62.0
2,2,200,2,6.136887,142.5
3,2,200,4,5.394614,125.9
4,2,50,2,0.013625,37.3
5,2,50,4,0.082258,30.1
6,2,500,2,71.669490,305.9
7,2,500,4,57.007970,305.5
8,3,100,2,0.805384,58.5
9,3,100,4,3.557904,48.7


In [28]:
## merge:
results_merge = results_limsbs.merge(IMSBS_group, on=["m", "n", "sigma"])
results_merge

,m,n,sigma,time_limsbs,quality_limsbs,time_imsbs,quality_imsbs
0,2,100,2,0.629813,72.4,0.199895,72.9
1,2,100,4,0.730760,62.0,0.266158,61.9
2,2,200,2,6.136887,142.5,1.242103,139.3
3,2,200,4,5.394614,125.9,1.633061,125.5
4,2,50,2,0.013625,37.3,0.006509,37.7
5,2,50,4,0.082258,30.1,0.036528,30.1
6,2,500,2,71.669490,305.9,15.176970,282.1
7,2,500,4,57.007970,305.5,12.263822,303.0
8,3,100,2,0.805384,58.5,0.358098,58.5
9,3,100,4,3.557904,48.7,1.135169,48.5


In [29]:
results_merge_final = results_merge[["m", "n", "sigma", "quality_limsbs", "quality_imsbs"] ]


In [30]:
results_merge_final.to_latex("results_merge_final.tex", index=False)

/tmp/ipykernel_210746/2296600886.py:1: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  results_merge_final.to_latex("results_merge_final.tex", index=False)


In [31]:
#results_merge_final.csv...
results_merge_final[ (results_merge_final["m"] == "2") | (results_merge_final["m"] == "3") ].mean()

m                 1.388889e+14
n                 6.256263e+41
sigma             1.515152e+14
quality_limsbs    1.150000e+02
quality_imsbs     1.127750e+02
dtype: float64

In [32]:
results_merge_final[ (results_merge_final["m"] == "5") | (results_merge_final["m"] == "10") ].mean()

m                 3.472222e+22
n                 6.256263e+41
sigma             1.515152e+14
quality_limsbs    2.364375e+01
quality_imsbs     2.158750e+01
dtype: float64

In [33]:
results_merge_final.mean()

m                 6.944444e+37
n                 3.128131e+85
sigma             7.575758e+29
quality_limsbs    6.932188e+01
quality_imsbs     6.718125e+01
dtype: float64

## REAL instances 

In [34]:
# LIMSBS-2000_imbs-iters-5000_num-roots-10_U10_5_5_A2_F2_real.csv IMSBS-2000_heuristic-h5_imbs-iters-5000_num-roots-10-ratvirus.csv

In [35]:
import pandas as pd

# load data
imsbs = pd.read_csv("tuned_results/REAL/IMSBS-2000_heuristic-h5_imbs-iters-5000_num-roots-10-ratvirus.csv")
limsbs = pd.read_csv("tuned_results/REAL/LIMSBS-2000_imbs-iters-5000_num-roots-10_U10_5_5_A2_F2_real.csv")

# rename columns to avoid _x / _y
imsbs = imsbs.rename(columns={
    "time": "time_imsbs",
    "quality": "quality_imsbs"
})

limsbs = limsbs.rename(columns={
    "time": "time_limsbs",
    "quality": "quality_limsbs"
})


In [36]:
# merge (SQL INNER JOIN)
merged = pd.merge(
    limsbs,
    imsbs,
    on="file",
    how="inner",
    validate="one_to_one"   # ensures no duplicates
)


In [37]:
merged['name']=merged.apply(lambda x: x["file"].split(".")[0].strip(), axis=1)
merged['type']=merged.apply(lambda x: x["file"].split(".")[1].strip(), axis=1)


In [38]:
merged["n"] = merged.apply(lambda x: x["file"].split("_")[1].strip(), axis=1)

In [39]:
merged = merged.drop(columns=["file"] )
merged = merged[["n", "type", "quality_limsbs", "time_limsbs",  "quality_imsbs", "time_imsbs"]]

## Convert to latex:

In [40]:
def highlight_best(row):
    if row["quality_limsbs"] > row["quality_imsbs"]:
        row["quality_limsbs"] = f"\\textbf{{{row['quality_limsbs']}}}"
    elif row["quality_limsbs"] < row["quality_imsbs"]:
        row["quality_imsbs"] = f"\\textbf{{{row['quality_imsbs']}}}"
    else:
        row["quality_limsbs"] = f"\\textbf{{{row['quality_limsbs']}}}"
        row["quality_imsbs"] = f"\\textbf{{{row['quality_imsbs']}}}"
    return row

latex_df = merged.copy()
merged_bold = latex_df.apply(highlight_best, axis=1)


merged_bold.to_latex(
    "real_results.tex",
    index=False,
    float_format="%.2f",
    caption="Comparison of LIMSBS and IMSBS on real instances.",
    label="tab:real_results",
    bold_rows=False,
    longtable=False
)



/tmp/ipykernel_210746/1126199371.py:15: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  merged_bold.to_latex(
